# Dual Language Translator: Part 1 English -> French
###  `Problem statement: Dual Language Translator: Description: Make a machine learning model with a feature that translates English words into both French and Hindi simultaneously. This feature should only translate English words or lines that have 10 or more letters. If an English word has fewer than 10 letters, the model should prompt the user to “upload again.” Guidelines: You have to train your own machine learning model. You should have a GUI for this task. The GUI should include an input section for entering English words and an output section for displaying the translated French and Hindi words.`

# Import the libraries

In [1]:
import collections
import json
import numpy as np
 
from keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences
from keras.models import Model, Sequential
from keras.layers import Input, Dense, Embedding, GRU, LSTM, Bidirectional, Dropout, Activation, TimeDistributed, RepeatVector
from keras.optimizers import Adam
from keras.losses import sparse_categorical_crossentropy

# Load the Dataset


In [ ]:
# Load Data
def load_data(path):
    input_file = path
    with open(input_file, "r") as f:
        data= f.read()
    return data.split('\n')

english_sentences = load_data("eng_data.txt")
french_sentences = load_data("french_data.txt")

In [17]:
print("\nEnglish Sentences",english_sentences[:1], "\nFrench Sentences: ",french_sentences[:1])
print("\nEnglish Sentences",english_sentences[1:2], "\nFrench Sentences: ",french_sentences[1:2])
print("\nEnglish Sentences",english_sentences[2:3], "\nFrench Sentences: ",french_sentences[2:3])


English Sentences ['new jersey is sometimes quiet during autumn , and it is snowy in april .'] 
French Sentences:  ["new jersey est parfois calme pendant l' automne , et il est neigeux en avril ."]

English Sentences ['the united states is usually chilly during july , and it is usually freezing in november .'] 
French Sentences:  ['les états-unis est généralement froid en juillet , et il gèle habituellement en novembre .']

English Sentences ['california is usually quiet during march , and it is usually hot in june .'] 
French Sentences:  ['california est généralement calme en mars , et il est généralement chaud en juin .']


## Preprocess th dataset

In [6]:
def tokenize(x):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(x)
    return tokenizer.texts_to_sequences(x), tokenizer

In [7]:
def pad(x, length=None):
    if length is None:
        length = max([len(sentence) for sentence in x ])
    return pad_sequences(x, maxlen=length, padding='post')

In [8]:
def preprocess(x,y):
    preprocess_x, x_tk = tokenize(x)
    preprocess_y, y_tk = tokenize(y)
    
    preprocess_x = pad(preprocess_x)
    preprocess_y = pad(preprocess_y)
    
    preprocess_y = preprocess_y.reshape(*preprocess_y.shape, 1)
    
    return preprocess_x, preprocess_y, x_tk, y_tk

preproc_english_sentences, preproc_french_sentences, english_tokenizer, french_tokenizer = preprocess(english_sentences, french_sentences)

max_english_sequence_length = preproc_english_sentences.shape[1]
max_french_sequence_length = preproc_french_sentences.shape[1]
english_vocab_size = len(english_tokenizer.word_index)
french_vocab_size = len(french_tokenizer.word_index) + 2 

print('Data Preprocessed')
print("Max English sentence length:", max_english_sequence_length)
print("Max French sentence length:", max_french_sequence_length)
print("English vocabulary size:", english_vocab_size)
print("French vocabulary size:", french_vocab_size)

Data Preprocessed
Max English sentence length: 15
Max French sentence length: 21
English vocabulary size: 199
French vocabulary size: 347


## Build and Train the model 

In [9]:
def bidirectional_embed_model(input_shape, output_sequence_length, english_vocab_size, french_vocab_size):
    
    # Hyperparameters
    learning_rate = 0.005
    
    # Build the model
    model = Sequential()
    model.add(Embedding(english_vocab_size+1, 256, input_length=input_shape[1], input_shape=input_shape[1:]))
    model.add(Bidirectional(GRU(256, return_sequences = True)))
    model.add(TimeDistributed(Dense(1024, activation ='relu')))
    model.add(Dropout(0.5))
    model.add(TimeDistributed(Dense(french_vocab_size, activation = 'softmax')))
     
    # Compile
    model.compile(loss = sparse_categorical_crossentropy,
                 optimizer = Adam(learning_rate),
                 metrics = ['accuracy'])
    return model

In [10]:
tmp_x = pad(preproc_english_sentences, max_french_sequence_length)
tmp_x = tmp_x.reshape((-1, preproc_french_sentences.shape[-2]))

In [11]:
# Build the model
embed_rnn_model = bidirectional_embed_model(
    tmp_x.shape,
    max_french_sequence_length,
    english_vocab_size,
    french_vocab_size)

embed_rnn_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 21, 256)           51200     
                                                                 
 bidirectional (Bidirectiona  (None, 21, 512)          789504    
 l)                                                              
                                                                 
 time_distributed (TimeDistr  (None, 21, 1024)         525312    
 ibuted)                                                         
                                                                 
 dropout (Dropout)           (None, 21, 1024)          0         
                                                                 
 time_distributed_1 (TimeDis  (None, 21, 347)          355675    
 tributed)                                                       
                                                        

In [12]:
embed_rnn_model.fit(tmp_x, 
                    preproc_french_sentences, 
                    batch_size=1024, 
                    epochs = 10, 
                    validation_split=0.2)

Epoch 1/10
108/108 [==============================] - 115s 667ms/step - loss: 1.3484 - accuracy: 0.6979 - val_loss: 0.4210 - val_accuracy: 0.8651
Epoch 2/10
108/108 [==============================] - 58s 534ms/step - loss: 0.3104 - accuracy: 0.9013 - val_loss: 0.1951 - val_accuracy: 0.9382
Epoch 3/10
108/108 [==============================] - 57s 526ms/step - loss: 0.1819 - accuracy: 0.9427 - val_loss: 0.1359 - val_accuracy: 0.9570
Epoch 4/10
108/108 [==============================] - 57s 524ms/step - loss: 0.1285 - accuracy: 0.9598 - val_loss: 0.1005 - val_accuracy: 0.9691
Epoch 5/10
108/108 [==============================] - 57s 528ms/step - loss: 0.1015 - accuracy: 0.9683 - val_loss: 0.0875 - val_accuracy: 0.9736
Epoch 6/10
108/108 [==============================] - 58s 537ms/step - loss: 0.0855 - accuracy: 0.9737 - val_loss: 0.0761 - val_accuracy: 0.9772
Epoch 7/10
108/108 [==============================] - 58s 540ms/step - loss: 0.0739 - accuracy: 0.9771 - val_loss: 0.0687 - val_a

## Save the model and the tokenizers

In [13]:
embed_rnn_model.save('english_to_french_model')

# Serialize English Tokenizer JSON
with open('english1_tokenizer.json', 'w', encoding = "utf8") as f:
    f.write(json.dumps(english_tokenizer.to_json(), ensure_ascii=False))
    

# Serialzie French Tokenizer JSON
with open('french_tokernizer.json', 'w', encoding ='utf8') as f:
    f.write(json.dumps(french_tokenizer.to_json(), ensure_ascii = False))
    

# Save max lengths
max_french_sequence_length_json = max_french_sequence_length
with open('sequence_length.json', 'w', encoding = 'utf8')as f:
    f.write(json.dumps(max_french_sequence_length_json, ensure_ascii = False))

INFO:tensorflow:Assets written to: english_to_french_model\assets


INFO:tensorflow:Assets written to: english_to_french_model\assets


## Make predictions

In [28]:
def logits_to_text(logits, tokenizer):
    index_to_words = {id : word for word, id in tokenizer.word_index.items()}
    index_to_words[0] = '<PAD>'
    
    return ' '.join([index_to_words[prediction] for prediction in np.argmax(logits, 1)])

In [33]:
print("Prediction: ")
print(logits_to_text(embed_rnn_model.predict(tmp_x[:1])[0], french_tokenizer))


print("\n Original Text : ")
print(english_sentences[:1])

print("\n Correct Translation : ")
print(french_sentences[:1])



Prediction: 
1/1 [==============================] - 0s 78ms/step
new jersey est parfois calme pendant l' automne et il est neigeux en avril <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD>

 Original Text : 
['new jersey is sometimes quiet during autumn , and it is snowy in april .']

 Correct Translation : 
["new jersey est parfois calme pendant l' automne , et il est neigeux en avril ."]
